# Figure S4

Draws GNN training, temporal-test, spatial-test, and prediction diagnostics.


In [1]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'outputs' / 'GNN_H1').exists():
            return candidate
    raise FileNotFoundError('Could not find outputs/GNN_H1 from the current working directory.')


def display_path(path: Path) -> str:
    try:
        return str(path.resolve().relative_to(ROOT))
    except ValueError:
        return str(path)


ROOT = find_repo_root()
OUT_DIR = ROOT / 'outputs' / 'figures' / 'FigS4'
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_SPECS = [
    {'interval_months': 1, 'seed': 11},
    {'interval_months': 3, 'seed': 33},
    {'interval_months': 6, 'seed': 11},
]
SPLIT_SPECS = [
    {'split': 'train', 'label': 'Training', 'color': '#56657A', 'output': 'FigS4_GNN_training.png'},
    {'split': 'test_temporal', 'label': 'Temporal test', 'color': '#2A9D8F', 'output': 'FigS4_GNN_temporal_test.png'},
    {'split': 'test_spatial', 'label': 'Spatial test', 'color': '#CF6228', 'output': 'FigS4_GNN_spatial_test.png'},
]
METRICS_CSV = OUT_DIR / 'FigS4_GNN_prediction_metrics.csv'
AXIS_LIMITS = (-5.0, 5.0)
AXIS_TICKS = [-5, -3, -1, 1, 3, 5]
HISTOGRAM_EDGES = np.linspace(-5.0, 5.0, 31)
EXPORT_DPI = 600

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans', 'sans-serif'],
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 15,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'savefig.dpi': EXPORT_DPI,
    'savefig.bbox': 'tight',
})

print('Output:', display_path(OUT_DIR))


Output: outputs\figures\FigS4


In [2]:
REQUIRED_COLUMNS = {'split', 'y_true_delta_h_m', 'y_pred_delta_h_m'}
prediction_tables = {}

for spec in MODEL_SPECS:
    horizon = spec['interval_months']
    seed = spec['seed']
    path = (
        ROOT / f'outputs/GNN_H{horizon}' / f'seed{seed}' /
        'predictions' / 'aem_gnn_predictions_all_splits.csv'
    )
    table = pd.read_csv(path)
    missing = REQUIRED_COLUMNS - set(table.columns)
    if missing:
        raise KeyError(f'{display_path(path)} is missing columns: {sorted(missing)}')
    prediction_tables[horizon] = table
    counts = table['split'].value_counts()
    print(
        f'H={horizon}, seed={seed}: '
        + ', '.join(f'{name}={int(counts.get(name, 0)):,}' for name in ['train', 'test_temporal', 'test_spatial'])
    )


H=1, seed=11: train=2,933, test_temporal=811, test_spatial=410
H=3, seed=33: train=3,059, test_temporal=1,085, test_spatial=687
H=6, seed=11: train=8,485, test_temporal=3,732, test_spatial=2,274


In [3]:
def calculate_metrics(observed: np.ndarray, predicted: np.ndarray) -> dict:
    observed = np.asarray(observed, dtype=np.float64)
    predicted = np.asarray(predicted, dtype=np.float64)
    finite = np.isfinite(observed) & np.isfinite(predicted)
    observed = observed[finite]
    predicted = predicted[finite]
    if observed.size < 2:
        raise ValueError('At least two finite observed/predicted pairs are required.')

    error = predicted - observed
    denominator = np.sum((observed - observed.mean()) ** 2)
    return {
        'n': int(observed.size),
        'rmse_m': float(np.sqrt(np.mean(error ** 2))),
        'mae_m': float(np.mean(np.abs(error))),
        'bias_m': float(np.mean(error)),
        'pearson_r': float(np.corrcoef(observed, predicted)[0, 1]),
        'spearman_rho': float(pd.Series(observed).corr(pd.Series(predicted), method='spearman')),
        'nse': float(1.0 - np.sum(error ** 2) / denominator),
    }


metric_rows = []
for model_spec in MODEL_SPECS:
    horizon = model_spec['interval_months']
    seed = model_spec['seed']
    table = prediction_tables[horizon]
    for split_spec in SPLIT_SPECS:
        split = split_spec['split']
        subset = table.loc[table['split'].eq(split)]
        values = calculate_metrics(
            subset['y_true_delta_h_m'].to_numpy(),
            subset['y_pred_delta_h_m'].to_numpy(),
        )
        metric_rows.append({
            'interval_months': horizon,
            'seed': seed,
            'prediction_column': f'y_pred_seed_{seed}',
            'split': split,
            'split_label': split_spec['label'],
            **values,
        })

prediction_metrics = pd.DataFrame(metric_rows)
prediction_metrics.to_csv(METRICS_CSV, index=False)
print('Saved:', display_path(METRICS_CSV))
display(prediction_metrics)


Saved: outputs\figures\FigS4\FigS4_GNN_prediction_metrics.csv


,interval_months,seed,prediction_column,split,split_label,n,rmse_m,mae_m,bias_m,pearson_r,spearman_rho,nse
0,1,11,y_pred_seed_11,train,Training,2933,0.397026,0.209620,-0.012013,0.642112,0.675022,0.411761
1,1,11,y_pred_seed_11,test_temporal,Temporal test,811,0.314081,0.183226,-0.037058,0.676234,0.724403,0.444383
2,1,11,y_pred_seed_11,test_spatial,Spatial test,410,0.411194,0.269336,-0.062436,0.638273,0.663226,0.391424
3,3,33,y_pred_seed_33,train,Training,3059,0.482826,0.289214,-0.053735,0.869001,0.855368,0.751092
4,3,33,y_pred_seed_33,test_temporal,Temporal test,1085,0.583611,0.351455,-0.091261,0.773481,0.784253,0.574951
5,3,33,y_pred_seed_33,test_spatial,Spatial test,687,0.782617,0.477645,-0.036142,0.663269,0.717360,0.430097
6,6,11,y_pred_seed_11,train,Training,8485,0.557205,0.369515,-0.005641,0.908155,0.902965,0.823119
7,6,11,y_pred_seed_11,test_temporal,Temporal test,3732,0.820307,0.559891,0.049267,0.819112,0.851951,0.662939
8,6,11,y_pred_seed_11,test_spatial,Spatial test,2274,0.787756,0.518381,0.021831,0.811946,0.843276,0.637987


In [4]:
def style_box_axis(ax: plt.Axes) -> None:
    ax.grid(True, color='#D9D9D9', lw=0.55, alpha=0.58)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('#222222')
        spine.set_linewidth(0.8)
    ax.tick_params(
        axis='both', which='major', top=True, right=True,
        direction='out', length=4.0, width=0.8,
    )


def plot_split_diagnostics(split_spec: dict) -> Path:
    split = split_spec['split']
    split_label = split_spec['label']
    color = split_spec['color']

    fig, axes = plt.subplots(2, 3, figsize=(19.0, 10.5), dpi=EXPORT_DPI)
    fig.subplots_adjust(left=0.045, right=0.988, top=0.905, bottom=0.075, wspace=0.095, hspace=0.18)
    fig.suptitle(split_label, x=0.013, y=0.987, ha='left', va='top', fontsize=20, fontweight='bold')

    for column, model_spec in enumerate(MODEL_SPECS):
        horizon = model_spec['interval_months']
        seed = model_spec['seed']
        table = prediction_tables[horizon]
        subset = table.loc[table['split'].eq(split), ['y_true_delta_h_m', 'y_pred_delta_h_m']].dropna()
        observed = subset['y_true_delta_h_m'].to_numpy(dtype=np.float64)
        predicted = subset['y_pred_delta_h_m'].to_numpy(dtype=np.float64)

        metric = prediction_metrics.loc[
            prediction_metrics['interval_months'].eq(horizon)
            & prediction_metrics['seed'].eq(seed)
            & prediction_metrics['split'].eq(split)
        ].iloc[0]

        scatter_ax = axes[0, column]
        scatter_ax.scatter(
            observed,
            predicted,
            s=13,
            color=color,
            edgecolors='none',
            alpha=0.25,
            rasterized=True,
            zorder=2,
        )
        scatter_ax.plot(AXIS_LIMITS, AXIS_LIMITS, color='#222222', lw=1.0, zorder=3)
        scatter_ax.set_xlim(*AXIS_LIMITS)
        scatter_ax.set_ylim(*AXIS_LIMITS)
        scatter_ax.set_xticks(AXIS_TICKS)
        scatter_ax.set_yticks(AXIS_TICKS)
        scatter_ax.set_title(
            f"H = {horizon} {'month' if horizon == 1 else 'months'}",
            fontsize=16,
            fontweight='bold',
            pad=8,
        )
        scatter_ax.set_xlabel(r'Observed $\Delta h$ (m)')
        if column == 0:
            scatter_ax.set_ylabel(r'Predicted $\Delta h$ (m)')
        scatter_ax.text(
            0.04,
            0.96,
            f"r = {metric['pearson_r']:.2f}\nNSE = {metric['nse']:.2f}\nRMSE = {metric['rmse_m']:.2f} m",
            transform=scatter_ax.transAxes,
            ha='left',
            va='top',
            fontsize=11.5,
            color='#222222',
        )
        style_box_axis(scatter_ax)

        hist_ax = axes[1, column]
        hist_ax.hist(
            observed,
            bins=HISTOGRAM_EDGES,
            density=True,
            histtype='stepfilled',
            facecolor='#D0D0D0',
            edgecolor='#8A8A8A',
            linewidth=0.9,
            alpha=0.76,
            label='Observed',
        )
        hist_ax.hist(
            predicted,
            bins=HISTOGRAM_EDGES,
            density=True,
            histtype='step',
            color=color,
            linewidth=1.7,
            label='Predicted',
        )
        hist_ax.set_xlim(*AXIS_LIMITS)
        hist_ax.set_xlabel(r'$\Delta h$ (m)')
        if column == 0:
            hist_ax.set_ylabel('Density')
        hist_ax.legend(loc='upper right', frameon=False)
        style_box_axis(hist_ax)

    output_path = OUT_DIR / split_spec['output']
    fig.savefig(output_path, dpi=EXPORT_DPI, bbox_inches='tight', pad_inches=0.06)
    plt.close(fig)
    return output_path


In [5]:
figure_paths = [plot_split_diagnostics(spec) for spec in SPLIT_SPECS]

expected_outputs = {
    'FigS4_GNN_training.png',
    'FigS4_GNN_temporal_test.png',
    'FigS4_GNN_spatial_test.png',
}
if {path.name for path in figure_paths} != expected_outputs:
    raise AssertionError('Unexpected Fig. S4_1 figure output list.')

print('Figure outputs written to:')
for path in figure_paths:
    print(' ', display_path(path))


Figure outputs written to:
  outputs\figures\FigS4\FigS4_GNN_training.png
  outputs\figures\FigS4\FigS4_GNN_temporal_test.png
  outputs\figures\FigS4\FigS4_GNN_spatial_test.png


In [6]:
try:
    from IPython.display import Image, display
except ImportError:
    Image = None
    display = None

for path in figure_paths:
    print(display_path(path))
    if Image is not None:
        display(Image(filename=str(path)))


outputs\figures\FigS4\FigS4_GNN_training.png


outputs\figures\FigS4\FigS4_GNN_temporal_test.png


outputs\figures\FigS4\FigS4_GNN_spatial_test.png
